# 02 - DataFrames, esquemas, SQL y formatos

## Conexión con la teoría

Este notebook muestra que DataFrames y Spark SQL expresan planes sobre el mismo motor,
por qué un esquema explícito es útil y qué ventajas aporta Parquet frente a CSV.

In [ ]:
import os, shutil, subprocess

os.environ["JAVA_HOME"] = "/Users/beatrizsuch/java/temurin-17/Contents/Home"
os.environ["PATH"] = f'{os.environ["JAVA_HOME"]}/bin:' + os.environ["PATH"]

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("pontia-computacion-distribuida")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} | {spark.sparkContext.master}")
print(f"Paralelismo por defecto: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

## 1. Crear datos con un esquema explícito

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType, DoubleType, TimestampType
)
from datetime import datetime

schema = StructType([
    StructField("trip_id", LongType(), nullable=False),
    StructField("city", StringType(), nullable=False),
    StructField("fare", DoubleType(), nullable=True),
    StructField("event_time", TimestampType(), nullable=False),
])

rows = [
    (1, "Madrid", 12.5, datetime(2026, 1, 10, 8, 0)),
    (2, "Valencia", 18.0, datetime(2026, 1, 10, 9, 0)),
    (3, "Madrid", None, datetime(2026, 1, 11, 10, 0)),
    (4, "Sevilla", -2.0, datetime(2026, 1, 11, 11, 0)),
]

trips = spark.createDataFrame(rows, schema)
trips.printSchema()
trips.show()

El esquema informa a Spark antes de ejecutar: nombres, tipos y nulabilidad. Esto permite
validar expresiones y planificar sin descubrir la estructura fila a fila.

## 2. DataFrame API y SQL llegan al mismo motor

In [ ]:
trips.createOrReplaceTempView("trips")

result_df = (
    trips
    .filter(F.col("fare") > 0)
    .groupBy("city")
    .agg(F.round(F.avg("fare"), 2).alias("avg_fare"))
)

result_sql = spark.sql("""
    SELECT city, ROUND(AVG(fare), 2) AS avg_fare
    FROM trips
    WHERE fare > 0
    GROUP BY city
""")

result_df.show()
result_sql.show()

In [ ]:
print("PLAN DATAFRAME")
result_df.explain(mode="formatted")
print("\nPLAN SQL")
result_sql.explain(mode="formatted")

No buscamos que el texto de ambos planes sea idéntico carácter por carácter. La observación
importante es que las dos interfaces terminan en operadores físicos equivalentes del motor SQL.

## 3. CSV: texto sin tipos incorporados

In [ ]:
csv_path = DATA_DIR / "trips_csv"
trips.write.mode("overwrite").option("header", True).csv(str(csv_path))

csv_strings = spark.read.option("header", True).csv(str(csv_path))
csv_inferred = spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))
csv_explicit = spark.read.option("header", True).schema(schema).csv(str(csv_path))

print("Sin inferencia:")
csv_strings.printSchema()
print("Con inferencia:")
csv_inferred.printSchema()
print("Con esquema explícito:")
csv_explicit.printSchema()

`inferSchema` requiere inspeccionar datos y puede producir decisiones no deseadas. En pipelines
de producción suele ser preferible declarar y versionar el esquema.

## 4. Parquet conserva tipos y permite leer solo lo necesario

In [ ]:
parquet_path = DATA_DIR / "trips_parquet"
trips.write.mode("overwrite").parquet(str(parquet_path))

parquet_trips = spark.read.parquet(str(parquet_path))
query = parquet_trips.select("city", "fare").filter(F.col("fare") > 10)

query.printSchema()
query.explain(mode="formatted")
query.show()

En el `Scan parquet`, busca las columnas leídas y `PushedFilters`. Spark puede evitar leer
columnas que no se usan y trasladar ciertos filtros hacia la fuente.

## 5. Nulos y expresiones de columna

In [ ]:
clean = (
    trips
    .withColumn("fare_valid", F.coalesce("fare", F.lit(0.0)))
    .withColumn(
        "quality_status",
        F.when(F.col("fare").isNull(), "missing")
         .when(F.col("fare") <= 0, "invalid")
         .otherwise("valid"),
    )
)
clean.show()

## Preguntas de cierre

1. ¿Qué información conoce Spark gracias al esquema antes de leer todas las filas?
2. ¿Por qué inferir el esquema de CSV tiene coste?
3. ¿Qué operadores comparten SQL y DataFrame API?
4. ¿Qué evidencias de column pruning y predicate pushdown aparecen en el plan?

In [ ]:
spark.stop()